In [1]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random

# กำหนด Seed เพื่อให้ผลลัพธ์ข้อมูลออกมารูปแบบเดิมทุกครั้งที่รัน
np.random.seed(42)
random.seed(42)

# ==========================================
# 1. สร้าง dim_channels (Table 2)
# ==========================================
channels_data = {
    'channel_id': ['CH_01', 'CH_02', 'CH_03', 'CH_04', 'CH_05'],
    'channel_name': ['Booking.com', 'Agoda', 'Direct Web', 'Walk-in', 'Corporate'],
    'channel_type': ['OTA', 'OTA', 'Direct', 'Direct', 'Corporate'],
    'commission_model': ['Percentage', 'Percentage', 'Flat Fee', 'None', 'Net Rate'],
    'default_commission_rate': [0.18, 0.15, 0.0, 0.0, 0.0],
    'contract_owner': ['Sales_A', 'Sales_A', 'Sales_B', 'Sales_B', 'Sales_C']
}
dim_channels = pd.DataFrame(channels_data)

# ==========================================
# 2. สร้าง dim_rate_codes (Table 3)
# ==========================================
rates_data = {
    'rate_code_id': ['RC_RACK', 'RC_PROMO', 'RC_CORP'],
    'rate_name': ['Standard Rack Rate', 'Promotional Discount', 'Corporate Rate'],
    'is_commissionable': [True, True, False]
}
dim_rate_codes = pd.DataFrame(rates_data)

# ==========================================
# 3. สร้าง fact_bookings (Table 1)
# ==========================================
num_bookings = 8000
# --- ปรับปีเป็น 2025 ---
start_date = datetime(2025, 1, 1)
end_date = datetime(2025, 12, 31)
date_range = (end_date - start_date).days

bookings = []
for i in range(num_bookings):
    # 3.1 สุ่มวันที่ Check-in แบบยังไม่มีเวลา
    random_days = random.randint(0, date_range)
    base_check_in = start_date + timedelta(days=random_days)

    is_weekend = base_check_in.weekday() >= 4 # ศุกร์ (4), เสาร์ (5), อาทิตย์ (6)
    month = base_check_in.month
    is_peak = month in [11, 12, 1, 2] # ช่วง High Season

    # เพิ่มความสมจริงของเวลา Check-in (14:00 - 22:59)
    check_in_hour = random.randint(14, 22)
    check_in_minute = random.randint(0, 59)
    check_in_date = base_check_in + timedelta(hours=check_in_hour, minutes=check_in_minute)

    # 3.2 ความสัมพันธ์ H3 (วันหยุด = OTA เยอะ)
    if is_weekend:
        channel_id = np.random.choice(['CH_01', 'CH_02', 'CH_03', 'CH_04', 'CH_05'], p=[0.4, 0.3, 0.15, 0.05, 0.10])
    else:
        channel_id = np.random.choice(['CH_01', 'CH_02', 'CH_03', 'CH_04', 'CH_05'], p=[0.15, 0.15, 0.3, 0.1, 0.3])

    # 3.3 การผูก Rate Code
    if channel_id == 'CH_05':
        rate_code_id = 'RC_CORP'
    else:
        rate_code_id = np.random.choice(['RC_RACK', 'RC_PROMO'], p=[0.6, 0.4])

    # 3.4 Lead time & Booking Date
    if channel_id in ['CH_01', 'CH_02']:
        lead_time = random.randint(5, 60)
    elif channel_id == 'CH_03':
        lead_time = random.randint(1, 30)
    else:
        lead_time = random.randint(0, 10)

    # เพิ่มความสมจริงของเวลาจอง (00:00 - 23:59)
    booking_hour = random.randint(0, 23)
    booking_minute = random.randint(0, 59)
    booking_date = base_check_in - timedelta(days=lead_time) + timedelta(hours=booking_hour, minutes=booking_minute)

    # 3.5 Length of Stay & Check-out Date
    los = random.randint(1, 5)

    # เพิ่มความสมจริงของเวลา Check-out (06:00 - 12:00)
    check_out_hour = random.randint(6, 12)
    check_out_minute = random.randint(0, 59) if check_out_hour < 12 else 0 # ถ้าออกตอน 12 ให้เป็น 12:00 ตรง
    check_out_date = base_check_in + timedelta(days=los, hours=check_out_hour, minutes=check_out_minute)

    # 3.6 คำนวณ Gross Revenue
    base_rate = 2500 if is_peak else 1800
    if is_weekend: base_rate *= 1.2
    if rate_code_id == 'RC_PROMO': base_rate *= 0.7 # Promo ลด 30%
    if rate_code_id == 'RC_CORP': base_rate = 1500 # Corp ราคาคงที่

    gross_revenue = base_rate * los

    # 3.7 คำนวณ Commission
    comm_rate = dim_channels.loc[dim_channels['channel_id'] == channel_id, 'default_commission_rate'].values[0]
    is_comm = dim_rate_codes.loc[dim_rate_codes['rate_code_id'] == rate_code_id, 'is_commissionable'].values[0]

    if is_comm:
        commission_amount = gross_revenue * comm_rate
    else:
        commission_amount = 0.0

    # สมมติ Flat Fee ของ Direct Web ให้ Booking ละ 50 บาท
    if channel_id == 'CH_03':
        commission_amount = 50.0

    net_revenue = gross_revenue - commission_amount

    # 3.8 สถานะการจอง
    if channel_id in ['CH_01', 'CH_02']:
        status = np.random.choice(['Checked-Out', 'Cancelled', 'Confirmed'], p=[0.75, 0.18, 0.07])
    elif channel_id == 'CH_03':
        status = np.random.choice(['Checked-Out', 'Cancelled', 'Confirmed'], p=[0.77, 0.16, 0.07])
    else:
        status = np.random.choice(['Checked-Out', 'Cancelled'], p=[0.95, 0.05])

    if status == 'Cancelled':
        gross_revenue = 0
        commission_amount = 0
        net_revenue = 0

    bookings.append([
        f"BKG_{i+1:05d}", booking_date, check_in_date, check_out_date,
        channel_id, rate_code_id, gross_revenue, commission_amount, net_revenue, status
    ])

fact_bookings = pd.DataFrame(bookings, columns=[
    'booking_id', 'booking_date', 'check_in_date', 'check_out_date',
    'channel_id', 'rate_code_id', 'gross_room_revenue', 'commission_amount',
    'net_room_revenue', 'status'
])

# ==========================================
# 4. สร้าง fact_marketing_spend (Table 4)
# ==========================================
spend_data = []
current_date = start_date
spend_id = 1
while current_date <= end_date:
    daily_spend = random.uniform(500, 3000)
    clicks = int(daily_spend / random.uniform(10, 25))

    # ตัดเวลา Spend Data ที่ 23:59:59 ของแต่ละวัน
    spend_datetime = current_date + timedelta(hours=23, minutes=59, seconds=59)

    spend_data.append([
        f"SP_{spend_id:04d}", spend_datetime, 'CH_03',
        np.random.choice(['Google Ads', 'Facebook']),
        round(daily_spend, 2), clicks
    ])
    current_date += timedelta(days=1)
    spend_id += 1

fact_marketing_spend = pd.DataFrame(spend_data, columns=[
    'spend_id', 'spend_date', 'channel_id', 'platform', 'cost_amount', 'clicks'
])

# ==========================================
# 5. Export เป็น Excel File (แบ่ง Sheets)
# ==========================================
filename = "The_Azure_Stay_Dataset_2025.xlsx"
with pd.ExcelWriter(filename, engine='openpyxl') as writer:
    fact_bookings.to_excel(writer, sheet_name='fact_bookings', index=False)
    dim_channels.to_excel(writer, sheet_name='dim_channels', index=False)
    dim_rate_codes.to_excel(writer, sheet_name='dim_rate_codes', index=False)
    fact_marketing_spend.to_excel(writer, sheet_name='fact_marketing_spend', index=False)

print(f"Data Generation Successful! Saved to {filename} (Year: 2025)")

Data Generation Successful! Saved to The_Azure_Stay_Dataset_2025.xlsx (Year: 2025)
